# Fase C · Seis consultas SQL analíticas

Ejecutar después de Fase B. Cada tabla se obtiene de PostgreSQL.

> **Alcance:** análisis del archivo recibido. Todas sus fechas tienen día entre 1 y 12; no se conoce su cobertura completa ni se ha confirmado el CSV oficial. Se conservan filas idénticas por falta de un identificador de evento. `CANTIDAD` se suma; no equivale necesariamente al número de filas.

In [1]:
from pathlib import Path
import sys
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/proyecto.py').is_file()), None)
if RAIZ is None:
    raise RuntimeError('Abra el notebook desde la carpeta del proyecto o entrega/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
import pandas as pd
from IPython.display import display, Markdown
get_ipython().run_line_magic('matplotlib', 'inline')
from src.proyecto import extraer, transformar, conectar, cargar, consultar

from src.consultas import interpretar
motor = conectar()

## Q1. ¿Cómo cambia la cantidad registrada por mes y frente al mes anterior?

Incluye `LAG` para comparar meses consecutivos.

In [2]:
sql_1 = """WITH mensual AS (
 SELECT t.anio, t.mes, SUM(h.cantidad) AS cantidad,
        COUNT(DISTINCT t.fecha) AS dias_con_registro
 FROM p1_homicidios.hecho_homicidios h
 JOIN p1_homicidios.dim_tiempo t ON h.tiempo_id=t.tiempo_id
 GROUP BY t.anio,t.mes
), cambios AS (
 SELECT *, LAG(cantidad) OVER (ORDER BY anio,mes) AS cantidad_anterior
 FROM mensual
)
SELECT *, cantidad-cantidad_anterior AS cambio_absoluto,
 ROUND(100.0*(cantidad-cantidad_anterior)/NULLIF(cantidad_anterior,0),2) AS cambio_pct
FROM cambios ORDER BY anio,mes"""
q1 = consultar(motor, sql_1)
display(q1)

,anio,mes,cantidad,dias_con_registro,cantidad_anterior,cambio_absoluto,cambio_pct
0,2020,1,442,12,NaN,NaN,NaN
1,2020,2,393,12,442.0,-49.0,-11.09
2,2020,3,364,12,393.0,-29.0,-7.38
3,2020,4,379,12,364.0,15.0,4.12
4,2020,5,355,12,379.0,-24.0,-6.33
...,...,...,...,...,...,...,...
67,2025,8,481,12,442.0,39.0,8.82
68,2025,9,443,12,481.0,-38.0,-7.90
69,2025,10,481,12,443.0,38.0,8.58
70,2025,11,435,12,481.0,-46.0,-9.56


In [3]:
display(Markdown("**Interpretación:** " + interpretar(0, q1)))

**Interpretación:** El archivo contiene 72 meses y suma 31,746 unidades de CANTIDAD. El máximo mensual es 561 en 2022-01; el cambio calculado describe el archivo, cuya cobertura de días es anómala.

## Q2. ¿Qué departamentos concentran la mayor cantidad registrada?

Incluye `RANK` y participación sobre el total.

In [4]:
sql_2 = """WITH totales AS (
 SELECT u.cod_departamento,u.departamento,SUM(h.cantidad) AS cantidad
 FROM p1_homicidios.hecho_homicidios h
 JOIN p1_homicidios.dim_ubicacion u ON h.ubicacion_id=u.ubicacion_id
 GROUP BY u.cod_departamento,u.departamento
)
SELECT *, RANK() OVER (ORDER BY cantidad DESC) AS posicion,
 ROUND(100.0*cantidad/SUM(cantidad) OVER (),2) AS participacion_pct
FROM totales ORDER BY cantidad DESC,cod_departamento"""
q2 = consultar(motor, sql_2)
display(q2)

,cod_departamento,departamento,cantidad,posicion,participacion_pct
0,76,VALLE DEL CAUCA,5589,1,17.61
1,05,ANTIOQUIA,4347,2,13.69
2,11,BOGOTA D.C.,2621,3,8.26
3,19,CAUCA,2015,4,6.35
4,08,ATLANTICO,1728,5,5.44
5,13,BOLIVAR,1414,6,4.45
6,54,NORTE DE SANTANDER,1251,7,3.94
7,47,MAGDALENA,1154,8,3.64
8,52,NARIÑO,1119,9,3.52
9,25,CUNDINAMARCA,1008,10,3.18


In [5]:
display(Markdown("**Interpretación:** " + interpretar(1, q2)))

**Interpretación:** VALLE DEL CAUCA ocupa el primer lugar con 5,589 unidades (17.61% del archivo). Son cantidades absolutas; sin población y cobertura verificadas no representan tasas de riesgo.

## Q3. ¿Cómo se distribuye la cantidad por año, zona y sexo?

`GROUP BY` sobre tres dimensiones distintas: tiempo, ubicación y sexo.

In [6]:
sql_3 = """SELECT t.anio,u.zona,s.sexo,SUM(h.cantidad) AS cantidad
FROM p1_homicidios.hecho_homicidios h
JOIN p1_homicidios.dim_tiempo t ON h.tiempo_id=t.tiempo_id
JOIN p1_homicidios.dim_ubicacion u ON h.ubicacion_id=u.ubicacion_id
JOIN p1_homicidios.dim_sexo s ON h.sexo_id=s.sexo_id
GROUP BY t.anio,u.zona,s.sexo ORDER BY t.anio,u.zona,s.sexo"""
q3 = consultar(motor, sql_3)
display(q3)

,anio,zona,sexo,cantidad
0,2020,RURAL,FEMENINO,142
1,2020,RURAL,MASCULINO,1533
2,2020,URBANA,FEMENINO,260
3,2020,URBANA,MASCULINO,2763
4,2021,RURAL,FEMENINO,157
5,2021,RURAL,MASCULINO,1858
6,2021,URBANA,FEMENINO,266
7,2021,URBANA,MASCULINO,3126
8,2022,RURAL,FEMENINO,168
9,2022,RURAL,MASCULINO,1962


In [7]:
display(Markdown("**Interpretación:** " + interpretar(2, q3)))

**Interpretación:** La combinación más frecuente es 2025, URBANA, MASCULINO: 3,388 unidades. La consulta cruza tres dimensiones y conserva 31,746 unidades; las diferencias son descriptivas.

## Q4. ¿Qué armas o medios predominan y cómo se distribuyen por sexo?

JOIN explícito entre el hecho, contexto y sexo.

In [8]:
sql_4 = """SELECT c.arma_medio,s.sexo,SUM(h.cantidad) AS cantidad
FROM p1_homicidios.hecho_homicidios h
JOIN p1_homicidios.dim_contexto c ON h.contexto_id=c.contexto_id
JOIN p1_homicidios.dim_sexo s ON h.sexo_id=s.sexo_id
GROUP BY c.arma_medio,s.sexo ORDER BY cantidad DESC,c.arma_medio,s.sexo"""
q4 = consultar(motor, sql_4)
display(q4)

,arma_medio,sexo,cantidad
0,ARMA DE FUEGO,MASCULINO,22930
1,ARMA BLANCA / CORTOPUNZANTE,MASCULINO,5023
2,ARMA DE FUEGO,FEMENINO,1466
3,CONTUNDENTES,MASCULINO,1212
4,ARMA BLANCA / CORTOPUNZANTE,FEMENINO,616
5,CONTUNDENTES,FEMENINO,338
6,ARTEFACTO EXPLOSIVO/CARGA DINAMITA,MASCULINO,141
7,ARTEFACTO EXPLOSIVO/CARGA DINAMITA,FEMENINO,11
8,DRON / EXPLOSIVO,MASCULINO,8
9,ARMA DE FUEGO,SIN ESTABLECER,1


In [9]:
display(Markdown("**Interpretación:** " + interpretar(3, q4)))

**Interpretación:** ARMA DE FUEGO concentra 24,397 unidades (76.85%). La separación por sexo describe registros, sin establecer causalidad ni riesgo individual.

## Q5. ¿Qué relación existe entre las cantidades urbanas y rurales por municipio?

Los ceros representan ausencia de filas en esa zona del municipio.

In [10]:
sql_5 = """SELECT u.cod_departamento,u.departamento,u.cod_municipio,u.municipio,
 SUM(CASE WHEN u.zona='URBANA' THEN h.cantidad ELSE 0 END) AS urbana,
 SUM(CASE WHEN u.zona='RURAL' THEN h.cantidad ELSE 0 END) AS rural,
 SUM(h.cantidad) AS cantidad
FROM p1_homicidios.hecho_homicidios h
JOIN p1_homicidios.dim_ubicacion u ON h.ubicacion_id=u.ubicacion_id
GROUP BY u.cod_departamento,u.departamento,u.cod_municipio,u.municipio
ORDER BY cantidad DESC,u.cod_municipio"""
q5 = consultar(motor, sql_5)
display(q5)

,cod_departamento,departamento,cod_municipio,municipio,urbana,rural,cantidad
0,11,BOGOTA D.C.,11001,BOGOTA D.C.,2610,11,2621
1,76,VALLE DEL CAUCA,76001,CALI,2502,59,2561
2,08,ATLANTICO,08001,BARRANQUILLA,894,1,895
3,05,ANTIOQUIA,05001,MEDELLIN,776,37,813
4,13,BOLIVAR,13001,CARTAGENA DE INDIAS,721,11,732
...,...,...,...,...,...,...,...
931,85,CASANARE,85279,RECETOR,0,1,1
932,85,CASANARE,85325,SAN LUIS DE PALENQUE,0,1,1
933,91,AMAZONAS,91263,EL ENCANTO,0,1,1
934,97,VAUPES,97666,TARAIRA,1,0,1


In [11]:
display(Markdown("**Interpretación:** " + interpretar(4, q5)))

**Interpretación:** Se observan 936 municipios; BOGOTA D.C. reúne 2,621 unidades. La correlación de Pearson urbana-rural es 0.163; está afectada por tamaño municipal, cobertura y valores extremos.

## Q6. ¿Cómo se distribuyen las cantidades mensuales de cada departamento?

La cuadrícula completa departamento-mes evita excluir combinaciones sin filas. Los ceros representan ausencia en el archivo.

In [12]:
sql_6 = """WITH meses AS (
 SELECT DISTINCT anio,mes FROM p1_homicidios.dim_tiempo
), departamentos AS (
 SELECT DISTINCT cod_departamento,departamento FROM p1_homicidios.dim_ubicacion
), observado AS (
 SELECT u.cod_departamento,t.anio,t.mes,SUM(h.cantidad) AS cantidad
 FROM p1_homicidios.hecho_homicidios h
 JOIN p1_homicidios.dim_tiempo t ON h.tiempo_id=t.tiempo_id
 JOIN p1_homicidios.dim_ubicacion u ON h.ubicacion_id=u.ubicacion_id
 GROUP BY u.cod_departamento,t.anio,t.mes
)
SELECT d.cod_departamento,d.departamento,m.anio,m.mes,
 COALESCE(o.cantidad,0) AS cantidad
FROM departamentos d CROSS JOIN meses m
LEFT JOIN observado o ON o.cod_departamento=d.cod_departamento
 AND o.anio=m.anio AND o.mes=m.mes
ORDER BY d.cod_departamento,m.anio,m.mes"""
q6 = consultar(motor, sql_6)
display(q6)

,cod_departamento,departamento,anio,mes,cantidad
0,05,ANTIOQUIA,2020,1,64
1,05,ANTIOQUIA,2020,2,70
2,05,ANTIOQUIA,2020,3,71
3,05,ANTIOQUIA,2020,4,44
4,05,ANTIOQUIA,2020,5,55
...,...,...,...,...,...
2371,99,VICHADA,2025,8,0
2372,99,VICHADA,2025,9,1
2373,99,VICHADA,2025,10,0
2374,99,VICHADA,2025,11,0


In [13]:
display(Markdown("**Interpretación:** " + interpretar(5, q6)))

**Interpretación:** La cuadrícula contiene 2376 combinaciones departamento-mes; su mediana es 8.0 unidades. Los 281 ceros significan ausencia de filas en el archivo para esa combinación, no ausencia confirmada de homicidios.

In [14]:
motor.dispose()